In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
invoices = pd.read_excel("CRATES-2026\\VANS\\MARCH\\INITIAL-DATA\\MAR-1.xlsx", sheet_name="Crate Movement Report")
inv = pd.ExcelFile("CRATES-2026\\VANS\\MARCH\\INITIAL-DATA\\MAR-1.xlsx")

In [4]:
inv.sheet_names

['Crate Movement Report', 'Sheet1', 'Sheet3', 'Sheet4']

In [5]:
invoices.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [6]:
invoices.head()

,DATES,STATUS,A/C,Invoice No.,QTY
0,2026-03-01,Inv,55,55260088,47
1,2026-03-01,Inv,57,57260078,118
2,2026-03-01,Inv,62,62260080,122
3,2026-03-01,Inv,63,63260070,91
4,2026-03-01,Inv,82,82260070,47


In [7]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 760 entries, 0 to 759
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        760 non-null    datetime64[ns]
 1   STATUS       760 non-null    object        
 2   A/C          760 non-null    int64         
 3   Invoice No.  760 non-null    int64         
 4   QTY          760 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 29.8+ KB


In [8]:
try:
    credit_notes = pd.read_excel("CRATES-2026\\VANS\\MARCH\\INITIAL-DATA\\MAR-1.xlsx", sheet_name="Sheet1")
except:
    credit_notes = pd.DataFrame()
    print("Credit note file not found")

In [9]:
credit_notes.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [10]:
credit_notes.head()

,DATES,STATUS,A/C,Invoice No.,QTY
0,2026-03-01,CN,57,57260081,1
1,2026-03-01,CN,582,582260158,0
2,2026-03-01,CN,704,704260195,0
3,2026-03-01,CN,704,704260196,0
4,2026-03-01,CN,704,704260197,1


In [11]:
credit_notes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 616 entries, 0 to 615
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        616 non-null    datetime64[ns]
 1   STATUS       616 non-null    object        
 2   A/C          616 non-null    int64         
 3   Invoice No.  616 non-null    int64         
 4   QTY          616 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 24.2+ KB


In [12]:
credit_notes.tail()

,DATES,STATUS,A/C,Invoice No.,QTY
611,2026-03-01,CN,7089,708926068,0
612,2026-03-01,CN,7096,709626061,7
613,2026-03-01,CN,7096,709626063,0
614,2026-03-01,CN,7098,709826102,0
615,2026-03-01,CN,7098,709826103,0


In [13]:
# Define numeric columns for processing
numeric_columns = ['QTY']

In [14]:
# Clean column names first
credit_notes.columns = credit_notes.columns.str.strip()
invoices.columns = invoices.columns.str.strip()

# Define numeric columns for processing
numeric_columns = ['QTY']

# Ensure QTY is integer and strip whitespace from A/C
invoices['QTY'] = invoices['QTY'].astype(int)
credit_notes['QTY'] = credit_notes['QTY'].astype(int)
invoices['A/C'] = invoices['A/C'].astype(str).str.strip()
credit_notes['A/C'] = credit_notes['A/C'].astype(str).str.strip()

# Only consider credit notes with STATUS == 'CN'
credit_notes_cn = credit_notes[credit_notes['STATUS'] == 'CN']

# Remove invoice rows that have an exact match in credit notes (by A/C and QTY)
filtered_invoices = invoices.copy()
for idx, credit_row in credit_notes_cn.iterrows():
    match = (
        (filtered_invoices['A/C'] == credit_row['A/C']) &
        (filtered_invoices['QTY'] == credit_row['QTY'])
    )
    match_idx = filtered_invoices[match].index
    if not match_idx.empty:
        # Drop only the first matching invoice for this credit note
        filtered_invoices = filtered_invoices.drop(match_idx[0])

# Group remaining invoices by A/C and sum QTY
grouped_df = filtered_invoices.groupby(['A/C'], as_index=False)[numeric_columns].sum()

# Get latest dates from invoices
latest_dates = filtered_invoices.groupby('A/C')['DATES'].max().reset_index()

# Merge latest dates with grouped QTY
final_df = pd.merge(latest_dates, grouped_df, on='A/C', how='left')

print(grouped_df)

      A/C  QTY
0     100   33
1    1000  152
2    1003   65
3    1008   10
4    1009   52
..    ...  ...
600   963   10
601   982   45
602   989   45
603   996    3
604   997   76

[605 rows x 2 columns]


In [15]:
# Export to Excel
final_df.to_excel('CRATES-2026\\VANS\\MARCH\\FINAL-OUTPUT\\DATE-1-MAR-VANS-CRATES-INVOICES.xlsx', index=False)